# 04 — Exact automatic differentiation

**Objectives.** Form gradients, Jacobians, and Hessians; understand forward/reverse products; verify derivatives numerically.

For $f:\mathbb R^n\to\mathbb R^m$, the Jacobian is $J_{ij}=\partial f_i/\partial x_j$. Forward mode computes $Jv$ efficiently; reverse mode computes $w^T J$. Hessians describe scalar-objective curvature and appear in Newton/KKT systems.


### Editable lesson setup

This cell makes the checked-out repository available to the notebook without
installing it as a Python package. It searches the current directory and its
parents for `casadi_tutorial/`, adds that repository root to `sys.path`, and
enables IPython's `autoreload` extension. With autoreload enabled, changes to the
editable CSTR, UR5e, integration, plotting, and diagnostic modules are picked up
before later cells execute.

There is no mathematical transformation in this setup cell. Its role is to make
the relationship between notebook code and local source code explicit:

$$
\text{notebook import} \longrightarrow
\text{current repository source},
$$

rather than resolving an older installed wheel or editable package.


In [24]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Construct exact first- and second-order derivatives

The cell builds an MX residual vector, forms its Jacobian, and defines a
scalar least-squares objective. `ca.hessian` returns both the Hessian and
gradient of that scalar objective. All four expressions are collected in
one `Function`, ensuring that the values and derivatives are evaluated
from the same symbolic model.

The residual and objective are

$$
r(x)=\begin{bmatrix}x_0x_1\\
\sin(x_1)+x_2^2\end{bmatrix},
\qquad
\phi(x)=r(x)^\mathsf{T}r(x).
$$

CasADi constructs

$$
J_r(x)=\frac{\partial r}{\partial x},\qquad
\nabla\phi(x)=\frac{\partial\phi}{\partial x},\qquad
H_\phi(x)=\frac{\partial^2\phi}{\partial x^2}
$$

by algorithmic differentiation of the expression graph rather than by
numerical perturbation.


In [ ]:
import casadi as ca
import numpy as np

x = ca.MX.sym("x", 3)
residual = ca.vertcat(
    x[0] * x[1],
    ca.sin(x[1]) + x[2] ** 2,
)

# jacobian differentiates a vector expression and returns one row per output.
residual_jacobian = ca.jacobian(residual, x)

# hessian returns both the square Hessian and gradient of a scalar expression.
objective = ca.dot(residual, residual)
objective_hessian, objective_gradient = ca.hessian(objective, x)

derivative_function = ca.Function(
    "derivatives",
    [x],
    [
        residual,
        residual_jacobian,
        objective_gradient,
        objective_hessian,
    ],
    ["x"],
    ["residual", "jacobian", "gradient", "hessian"],
)
evaluation_point = np.array([1.0, 0.4, -0.2])
values = derivative_function(evaluation_point)
print("Jacobian:", values[1])
print("Hessian:", values[3])

### Verify a Jacobian-vector product with finite differences

This cell checks the exact directional derivative produced by CasADi
against a centered finite-difference approximation. The residual
expression is wrapped in a `Function` for numerical evaluation at points
displaced along a chosen direction $v$.

The exact directional derivative is

$$
D_vr(x)=J_r(x)v.
$$

The centered approximation used for validation is

$$
D_vr(x)\approx
\frac{r(x+\varepsilon v)-r(x-\varepsilon v)}{2\varepsilon},
\qquad \varepsilon=10^{-6}.
$$

Agreement validates the derivative graph while illustrating that finite
differences are useful as a check, not as the primary derivative method.


In [ ]:
direction = np.array([0.3, -0.1, 0.2])
finite_difference_step = 1e-6
residual_function = ca.Function("residual", [x], [residual])

# A centered finite difference checks the Jacobian-vector product J @ direction.
finite_difference = (
    residual_function(evaluation_point + finite_difference_step * direction)
    - residual_function(evaluation_point - finite_difference_step * direction)
) / (2 * finite_difference_step)
exact_directional_derivative = values[1] @ direction

np.testing.assert_allclose(
    finite_difference.full().ravel(),
    exact_directional_derivative.full().ravel(),
    rtol=1e-6,
    atol=1e-8,
)

**Common mistake:** finite differences can validate a smooth graph, but step-size cancellation makes them a poor replacement for AD. Nondifferentiable `fabs`, `fmax`, and conditionals require modeling judgment. **Recap:** choose derivatives by the product needed, and inspect their sparsity.
